# Lab 2 — LoRA fine-tuning of OpenVLA on a SageMaker Training Job

| | Serverless SFT lab | This lab |
|---|---|---|
| API | `SFTTrainer` (managed recipe) | `ModelTrainer` + `scripts/` |
| Compute | chosen by the service | single-L40S `ml.g6e` sizes (smoke) / 4-8 L40S (full), with fallback |
| Container | managed | PyTorch 2.7 training DLC + `scripts/requirements.txt` |
| Distribution | managed | Accelerate DDP, one process per GPU |
| Tracking | MLflow App | same MLflow App (`MLFLOW_TRACKING_URI` = app ARN) |
| Output | model package + `hf_merged/` | `adapter/` + `hf_merged/`, registered as a model package |

**Training recipe** (`scripts/recipes/openvla_lora.yaml`, from `openvla/vla-scripts/finetune.py`):
LoRA r=32, alpha=16, dropout 0, `all-linear` targets (~111 M trainable params, 1.4 %), AdamW lr 5e-4
cosine, bf16, 4 samples/GPU, global batch 32 (gradient accumulation fills the gap on fewer GPUs).
The loss covers only the 7 action tokens + EOS.

**Prerequisites:** a training quota for the instance types in `config.TRAIN` (`ml.g6e.xlarge for training job
usage` and the larger single-L40S sizes for the smoke run; `ml.g6e.12xlarge` and up for the full run).
`config.CAPACITY = "fallback"` (default) lets SageMaker start the job on the first type in the list that has
capacity; `"spot"` is cheaper but can queue for hours, so keep it for self-paced runs. `openvla/openvla-7b` is
public, so no Hugging Face token is needed.

**No GPU capacity, or short on time?** Skip this notebook and run `2b-import-pretrained.ipynb`, which imports
the artifacts of a previous training run so notebooks 3 and 4 work unchanged.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE}, capacity: {config.CAPACITY} -> {', '.join(config.TRAIN['instance_types'])}")
print(f"s3 root:  {s3_root}")

In [ ]:
# MLflow tracking (optional; metrics also appear in the job log either way)
#   MLFLOW_APP_ARN = "arn:aws:sagemaker:..."  -> use this MLflow App
#   MLFLOW_APP_ARN = ""                       -> reuse the workshop's app, or create one if the account has none
#                                                (creating takes several minutes, once)
#   USE_MLFLOW = False                        -> skip MLflow
USE_MLFLOW = config.USE_MLFLOW
MLFLOW_APP_ARN = config.MLFLOW_APP_ARN

from config import resolve_mlflow_app

mlflow_app_arn = resolve_mlflow_app(sm_client, session=sess, arn=MLFLOW_APP_ARN) if USE_MLFLOW else None
mlflow_env = {"MLFLOW_TRACKING_URI": mlflow_app_arn, "MLFLOW_EXPERIMENT_NAME": config.MLFLOW_EXPERIMENT} if mlflow_app_arn else {}
print(f"MLflow: {mlflow_app_arn or 'off'}")

## Configure the job

* `OutputDataConfig(compression_type="NONE")` writes the merged 15 GB checkpoint uncompressed, so it
  can be served straight from S3 (same idea as `hf_merged/` in the serverless lab).
* `CheckpointConfig` syncs `/opt/ml/checkpoints` (LoRA adapter + optimizer state) to S3. With managed Spot,
  SageMaker restarts an interrupted job and the script resumes from the latest checkpoint. The job may sit in
  `Starting` while it waits for Spot capacity (up to `SPOT_MAX_WAIT_H`).

In [ ]:
import time

from sagemaker.core.training.configs import CheckpointConfig, InputData, OutputDataConfig, SourceCode
from sagemaker.train import ModelTrainer

from job_utils import capacity_config

run_name = f"{config.PROJECT}-{config.RUN_MODE}-{time.strftime('%Y%m%d-%H%M%S')}"
compute, stopping_condition = capacity_config(
    config.TRAIN["instance_types"], config.CAPACITY, volume_size_in_gb=200,
    max_runtime_s=config.TRAIN["max_runtime_h"] * 3600, spot_max_wait_h=config.SPOT_MAX_WAIT_H)

trainer = ModelTrainer(
    training_image=config.TRAINING_IMAGE.format(region=region),
    source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script="run_finetuning.sh"),
    compute=compute,
    stopping_condition=stopping_condition,
    output_data_config=OutputDataConfig(s3_output_path=f"{s3_root}/training-output", compression_type="NONE"),
    checkpoint_config=CheckpointConfig(s3_uri=f"{s3_root}/checkpoints/{run_name}", local_path="/opt/ml/checkpoints"),
    environment={
        "RECIPE": config.TRAIN["recipe"],
        **mlflow_env,
        "NCCL_DEBUG": "WARN",
    },
    base_job_name=run_name[:48],
    role=role,
    sagemaker_session=sess,
)
print(f"run: {run_name}\nrecipe: {config.TRAIN['recipe']}\ncapacity: {config.CAPACITY} {config.TRAIN['instance_types']}\ndata: {dataset_s3_uri}")

In [ ]:
from job_utils import latest_job, print_phases, selected_instance, wait_for_job

## Train

Submitting returns immediately; the job then runs in SageMaker on its own.

In [ ]:
trainer.train(input_data_config=[InputData(channel_name="train", data_source=dataset_s3_uri)], wait=False)
job_name = trainer._latest_training_job.training_job_name
print(f"submitted: {job_name}")

### Wait for the job, or reconnect to it

`wait_for_job` prints the job's status and the latest `[step N]` line whenever they change (the full log is in
CloudWatch, linked below; the metrics are also in MLflow if enabled). You can stop this cell at any time: that
only stops the watching, not the job.

**Lost the connection or restarted the kernel?** The job keeps running. Run the first setup cell and the
`job_utils` import above, then this cell: it finds the most recent job for this run mode.

A run spends most of its time outside training: waiting for (Spot) capacity, pulling the container, installing
packages, downloading the 15 GB base model, then merging and uploading the 15 GB result. `print_phases` below
shows where the time went.

In [ ]:
job_name = globals().get("job_name") or latest_job(f"{config.PROJECT}-{config.RUN_MODE}")
job = wait_for_job(job_name)
print_phases(job)

In [ ]:
model_prefix = job["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/"
merged_s3_uri = model_prefix + "hf_merged/"

bucket, key = merged_s3_uri[5:].split("/", 1)
merged_files = [o["Key"].rsplit("/", 1)[-1] for o in s3.list_objects_v2(Bucket=bucket, Prefix=key)["Contents"]]
assert "config.json" in merged_files and "modeling_prismatic.py" in merged_files, merged_files
print(f"job:    {job_name} ({job['TrainingJobStatus']} on {selected_instance(job)}, {job['TrainingTimeInSeconds'] // 60} min training, "
      f"{job['BillableTimeInSeconds'] // 60} min billed)")
print(f"merged: {merged_s3_uri}")
print("\n".join(sorted(merged_files)))

## Register the model

The serving handler (`serving/inference.py`, plus `scripts/openvla_utils.py`) is stored next to the
weights under `hf_merged/code/`. The PyTorch inference container loads it from `/opt/ml/model/code`.
The package is registered as `PendingManualApproval`. Lab 3 approves it if it beats the base model.

In [ ]:
from artifacts import register_model

package = register_model(merged_s3_uri, source=job_name, run_mode=config.RUN_MODE, region=region)
print(f"model package: {package['ModelPackageArn']}")

Continue with **3-evaluation.ipynb**.